# CA4 - Telecom Customer Churn Classification

#### Gr 30: Sondre Aasheim & Zuzanna Magdalena Smolinska

Rules and context of the competition

A health research group wants to predict a person's cognitive assessment score (a simulated MoCA score) from self-reported lifestyle habits and a few biological measurements. Collecting all measurements for every participant is expensive. You will help them train a machine learning model for that task.

ca5bilde.png

1. The general workflow is similar to that of the previous Kaggle competitions, although regression is now the focus of the assignment.

2. You have access to a wide range of models and preprocessing tools taught in lectures. All models presented in the lecture are allowed. Models that were not presented in the lecture are not allowed. Use the appropriate scikit-learn tools to preprocess and automatically search for optimal parameter values.

3. You may try to engineer new features if you wish to do so. If you are unsure how to structure your notebook follow the steps: Imports, Reading the data, Data exploration and visualization, Data cleaning, Data exploration and visualization after cleaning, Data preprocessing (and possibly some more visualization), Evaluating different models/hyperparameters (using cross-validation), Kaggle submission (with prediction and result on the test set).

4. Bonus task (not mandatory): Measuring features is expensive. The company would like to measure as few features as possible. What is the minimum amount of original features (and which features) you can use without sacrificing too much performance? The company is willing to accept an MAE of 10% less than the MAE of your best model.

5. Your submission must use the following components/techniques:
Hyperparameter tuning
Scikit-learn Pipelines
Cross-validation
The Mean Absolute Error (MAE) as the performance metric (this is what is scored on Kaggle).
Moreover, you must build and evaluate at least two of the following three model pipelines (A-C). All pipelines should contain suitable preprocessing. At least one of your pipelines has to use a model based on linear regression (one of the linear regression-based regressors covered in the lecture, e.g. multi-linear regression/polynomial regression/PCR/PLS/RANSAC/regularized regression...).

(A) Regression analysis.

(B) Multi-class classification analysis with an ensemble classifier. To this end bin the target variable into discrete chunks and train a classification model on the binned target variable. For binning continuous data (transforming continuous target to discrete target values), you can use pandas.qcut() or pandas.cut() (for example code look at the end of this task description). Test and comment on how the number of bins affects the prediction.

(C) A two-step analysis (two sequential pipelines). In the first step, train an ensemble classification model to separate participants who score the maximum of 30 (the ceiling of the scale) from participants who score below 30. In the second step, use a regression model to estimate the cognitive assessment score of those samples that the classifier identifies as scoring below 30. Combine the results of both steps into a single prediction vector.

If your best model is not one of the two mandatory pipelines, also provide the code for training/evaluation of your best model (that means you provide code for three models in this case).

### 1: Imports

In [ ]:
import numpy as np                      # for numerical operations
import pandas as pd                     # for reading the csv files and working with dataframes
import math
import matplotlib.pyplot as plt         # for basic plotting
import seaborn as sns                   # for more advanced plotting

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from scipy.special import comb                             
from sklearn.ensemble import BaggingClassifier 
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.pipeline      import make_pipeline
from sklearn.ensemble import VotingClassifier
from sklearn.impute import SimpleImputer

from io import StringIO


Data Description:

The data is synthetic (generated with Python) and simulates a lifestyle and health survey of adults. It contains no real patient data. The target is a simulated score modeled after the Montreal Cognitive Assessment (MoCA).

 

Demographics & Biometrics

id: Identifier of the participant. It carries no information and must not be used as a feature, but it is needed in the submission file.

age: The age of the participant (18 to 90).

blood_pressure_sys: Systolic blood pressure (the top number). Influenced by age, diet, stress and exercise within the simulation.

 


Core Lifestyle Habits

sleep_duration_hrs: Average hours of sleep per night. Negatively impacted by high stress levels in the simulation.

sleep_quality_score: Self-rated scale from 1 (poor) to 10 (excellent).

cardio_mins_wk: Total minutes of elevated heart rate exercise per week. Follows a log-normal distribution (most do a little, a few do a lot).

water_intake_liters: Average daily water consumption. Note: included as a "noise" variable for feature selection practice.

screen_time_recreational_hrs: Daily hours spent on non-work screens.

social_media_hrs: Daily hours specifically on social media apps.

 

Diet & Substance Intake

diet_ultra_processed_pct: Estimated percentage of the diet consisting of highly processed foods.

diet_omega3_freq: Days per week of consuming brain-healthy foods like fish, nuts, or seeds.

smoking_status: Categorical feature with the values "Current", "Former" or "Never".

alcohol_drinks_wk: Total alcoholic beverages consumed per week.

 

Neuroplasticity & Mental Health

novel_learning_hrs_wk: Hours per week spent learning a new, difficult skill (e.g. a language, coding, an instrument). Acts as a neuroprotective bonus in the target calculation.

social_connection_score: Self-rated scale from 1 (isolated) to 10 (highly supported).

chronic_stress_score: Self-rated scale from 1 (low) to 10 (high) on perceived daily stress. Acts as a heavy penalty on the target.

 

Target

The cognitive assessment score (integer, documented range 5 to 30). TARGET. It is a baseline score minus penalties (age, stress, poor diet, smoking) plus bonuses (cardio, learning, social connection).

### 2: Reading the data

In [ ]:
"""
Reading the data -
both files must be in the same folder as this notebook
"""
# Montreal Cognitive Assessment = MoCA
MoCA_train = pd.read_csv("train.csv")
MoCA_test = pd.read_csv("test.csv")


### 3: Data exploration & visualisation

In [ ]:
# printing the size of each dataset
print("train file shape:", MoCA_train.shape)
print("test file shape: ", MoCA_test.shape)

In [ ]:
# printing the first 5 rows of the train dataset to get a look at the dataframe  
MoCA_train.head()


In [ ]:
# printing summary statistics for every numeric column, to get an overall view over the data, 
MoCA_train.describe()

In [ ]:
# histograms of all numeric features to see how the values are distributed

MoCA_train.hist(figsize=(12, 10), bins=20)
plt.tight_layout()          # adjusts spacing so the titles and axes don't overlap
plt.show()

In [ ]:
# printing the first 5 rows of the train dataset to get a look at the dataframe  
MoCA_train.head()

In [ ]:
"""
We build correlation matrix: how strongly the features are related to each other and to the target.
id (just an ID) is left out.

Correlation only works on numbers, so we turn "curent"/"former"/"never" into 2/1/0 so that the smoking can be included.
"""

# Turns smoking_status into numerical values instead of just words
MoCA_train["smoking_status"] = MoCA_train["smoking_status"].map({
    "Never": -1,
    "Former": 0,
    "Current": 1        # Creates a positiv number for those that do smoke
})

corr_data = MoCA_train.drop(columns=["id"]).copy()

corr = corr_data.corr()
plt.figure(figsize=(14, 12))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation matrix")
plt.show()

### 4: Data cleaning 

In [ ]:
# counting all missing values in the training and test data
# this should give 274393

print(MoCA_train.isna().sum().sum())
print(MoCA_test.isna().sum().sum())

In [ ]:
# First checking the amount of rows removed if the rows with NaN get removed outright. 
MoCA_train_dropNA = MoCA_train.dropna(axis=0)

print("train file shape:", MoCA_train.shape)
print("removed NaN shape:", MoCA_train_dropNA.shape)


 Discussion: 

A removeal of almost half the dataset is far too much, even if it keeps the amount of data we need to go through smaller as requested by the company. 

In [ ]:
# !!! Can come back to test replacements for "mean"
# Impute missing values using the column mean
imputer = SimpleImputer(missing_values=np.nan, strategy="mean") # Other popular choices: "median", "most_frequent"
imputer.fit(MoCA_train.values)
imputed_data = imputer.transform(MoCA_train.values)

# SimpleImputer converts the dataset to a NumPy array so we need to convert it back to a 
# pandas DataFrame to use our helper function
MoCA_train_dropNA = pd.DataFrame(imputed_data)

# Changes the names for the feautres back to their orginial names 
MoCA_train_dropNA.columns = ['id', 'age', 'sleep_duration_hrs', 'sleep_quality_score', 
'cardio_mins_wk', 'water_intake_liters', 'screen_time_recreational_hrs', 'social_media_hrs', 
'blood_pressure_sys', 'diet_ultra_processed_pct', 'diet_omega3_freq', 'smoking_status', 
'alcohol_drinks_wk', 'novel_learning_hrs_wk', 'social_connection_score', 'chronic_stress_score', 'target']
MoCA_train_dropNA.head()

In [ ]:
# Removes the id, as that has no impact on the study itself 
MoCA_train = MoCA_train.drop(columns=["id"]).copy()

In [ ]:
numeric_features = ["age", "sleep_duration_hrs", "sleep_quality_score", "cardio_mins_wk", "water_intake_liters",
                    "screen_time_recreational_hrs", "social_media_hrs", "blood_pressure_sys", "diet_ultra_processed_pct",
                    "diet_omega3_freq", "smoking_status", "alcohol_drinks_wk", "novel_learning_hrs_wk", 
                    "social_connection_score", "chronic_stress_score", "target"]

# z = (x - mean) / std: how many standard deviations each value is from the mean
z_scores = (MoCA_train_dropNA[numeric_features] - MoCA_train_dropNA[numeric_features].mean()) /MoCA_train_dropNA[numeric_features].std(ddof=0)

# checking if the z score is above or under the average by 5 through using absolute values
# as 3 which is often used brought about a removal of almost 10 %
outliers = z_scores.abs() > 5

# showing only the features that do have outliers
print("Outliers per feature:")
print(outliers.sum()[outliers.sum() > 0])

In [ ]:
# checking the rows with at least one outlier, and whether these customers churned
outlier_rows = outliers.any(axis=1)

print("Rows with at least one outlier:", outlier_rows.sum(), "of", len(MoCA_train_dropNA))

In [ ]:
# Filtering the data
MoCA_train_clean = MoCA_train_dropNA.loc[~outlier_rows].copy()

### 5: Data exploration & visualisation after cleaning

In [ ]:
corr_data_clean = MoCA_train_clean

corr = corr_data_clean.corr()
plt.figure(figsize=(14, 12))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation matrix")
plt.show()

KeyboardInterrupt: 

Discussion:

the removal of the outliers only seem to have strengthend most all of the trends

### 6: Preprocessing

In [ ]:
"""
Preparing the data to be used for modelling:
- X:      the features of the training customers (everything except Churn)
- y:      the target we want to predict (0 = stays, 1 = leaves)
- X_test: the features of the Kaggle test customers, prepared in exactly the same way as X
"""

X = MoCA_train_clean.drop(columns=["target"])
y = MoCA_train_clean["target"]
X_test = preprocess(telecom_customer_test)

# checking the result: 
# X and X_test must have the same number of columns,
# X and y must have the same number of rows

print("X:     ", X.shape)
print("y:     ", y.shape)
print("X_test:", X_test.shape)

# the first rows of the prepared data
X.head()


Rules and context of the competition

A health research group wants to predict a person's cognitive assessment score (a simulated MoCA score) from self-reported lifestyle habits and a few biological measurements. Collecting all measurements for every participant is expensive. You will help them train a machine learning model for that task.

ca5bilde.png

1. The general workflow is similar to that of the previous Kaggle competitions, although regression is now the focus of the assignment.

2. You have access to a wide range of models and preprocessing tools taught in lectures. All models presented in the lecture are allowed. Models that were not presented in the lecture are not allowed. Use the appropriate scikit-learn tools to preprocess and automatically search for optimal parameter values.

3. You may try to engineer new features if you wish to do so. If you are unsure how to structure your notebook follow the steps: Imports, Reading the data, Data exploration and visualization, Data cleaning, Data exploration and visualization after cleaning, Data preprocessing (and possibly some more visualization), Evaluating different models/hyperparameters (using cross-validation), Kaggle submission (with prediction and result on the test set).

4. Bonus task (not mandatory): Measuring features is expensive. The company would like to measure as few features as possible. What is the minimum amount of original features (and which features) you can use without sacrificing too much performance? The company is willing to accept an MAE of 10% less than the MAE of your best model.

5. Your submission must use the following components/techniques:
Hyperparameter tuning
Scikit-learn Pipelines
Cross-validation
The Mean Absolute Error (MAE) as the performance metric (this is what is scored on Kaggle).
Moreover, you must build and evaluate at least two of the following three model pipelines (A-C). All pipelines should contain suitable preprocessing. At least one of your pipelines has to use a model based on linear regression (one of the linear regression-based regressors covered in the lecture, e.g. multi-linear regression/polynomial regression/PCR/PLS/RANSAC/regularized regression...).

(A) Regression analysis.

(B) Multi-class classification analysis with an ensemble classifier. To this end bin the target variable into discrete chunks and train a classification model on the binned target variable. For binning continuous data (transforming continuous target to discrete target values), you can use pandas.qcut() or pandas.cut() (for example code look at the end of this task description). Test and comment on how the number of bins affects the prediction.

(C) A two-step analysis (two sequential pipelines). In the first step, train an ensemble classification model to separate participants who score the maximum of 30 (the ceiling of the scale) from participants who score below 30. In the second step, use a regression model to estimate the cognitive assessment score of those samples that the classifier identifies as scoring below 30. Combine the results of both steps into a single prediction vector.

If your best model is not one of the two mandatory pipelines, also provide the code for training/evaluation of your best model (that means you provide code for three models in this case).

### 7: Evaluating models/hyperparameters (cross-validation)

### 8: Kaggle submission

Deliverables / submission:

To have the compulsory assignment approved you need to deliver the following:

1. Data exploration and visualization of the data is mandatory and is a requirement for passing the compulsory. Use several techniques to explore the data effectively. It has to be evident from your code comments what you concluded from the data exploration.

2. Your name must appear on the leaderboard of our in-class Kaggle competition, which means that you must submit at least one prediction (link to the leaderboard: https://www.kaggle.com/competitions/dat-200-ca-5-h-2026/leaderboard). Read the description on Kaggle to see how your predictions will be evaluated.

3. We expect you to beat the benchmark (named BEAT ME) on the public leaderboard (performance metric: Mean Absolute Error)

4. Submit (I) a Jupyter notebook; (II) PDF of the Jupyter notebook; (III) Python code of your Jupyter notebook on Canvas: with the code for the training of your best model and its prediction on the test data that you submitted to Kaggle, with all mandatory parts listed in the rules, and with data exploration and visualization. Please make short comments throughout your notebook/code on what you are doing and how you choose the parameters of your final best model. Please clearly identify your best model and comment with the conclusion about which one it is. Leave out everything that is not necessary, keep only what is needed. We will reject submissions that are full of unnecessary stuff and output!!

5. If you use an alias (username that is not your real name) in the Kaggle leaderboard, you must provide your Kaggle alias and your real name at the beginning of your Jupyter notebook.

Notes

Important: You are responsible for uploading the submission to Canvas and Kaggle before the deadline. You need to plan for possible technical difficulties ahead. (You can submit multiple times before the deadline, but only the final submission counts.)

Important:  You only need what you learned in the lecture and the documentation pages of the used Python libraries to complete the assignment. Anything covered in the course could be relevant. If you copy some larger bit of code (multiple lines or some complex expression), cite the source (e.g. a link) and mark the part of the code that is copied and add an explanation of what the code is doing in your own words. Review the CA rules: /courses/14056/pages/rules-for-compulsory-assignments-ca. Remember, that you need to pass all CAs to be admitted to the exam.

 

Kaggle competition:
You will participate in an In-Class Kaggle competition (only students enrolled in the DAT200 course are allowed to participate), where you will compete with your fellow students on how well your model predicts the test data.
1. Here is the link to enter the competition (please don't forward this link to others outside our DAT200-course, since we want to keep the competition in-class):

https://www.kaggle.com/t/b1e9018c78be4579905cd7030450c672
 

2. On the "data"-Tab (https://www.kaggle.com/competitions/dat-200-ca-5-h-2026/data) of the In-Class competition you will find three files:

train.csv
test.csv
example_submission.csv
3. Notebook Template. You can use the attached notebook template from the previous compulsory assignments to build your pipelines (or start from scratch):

CA3.zipLast ned CA3.zip